# DizerCore LoRA Training — Qwen2.5-1.5B-Instruct

Train a TrinityCore-specific LoRA adapter on the free T4 GPU.

**Setup:**
1. Runtime → Change runtime type → T4 GPU
2. Dataset must be at `MyDrive/dizercore/dizercore-dataset.jsonl`
3. Click Runtime → Run all

Runtime: ~2 hours on T4.

In [ ]:
import subprocess, sys, os, gc

from google.colab import drive
drive.mount('/content/drive')

print('=== Install ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers', 'peft', 'bitsandbytes',
                'accelerate', 'datasets'], check=True)

# ---------- Config ----------
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
DATASET = '/content/drive/MyDrive/dizercore/dizercore-dataset.jsonl'
WORK = '/content'
ADAPTER = f'{WORK}/lora-adapter'
MERGED = f'{WORK}/dizercore-fp16'
GGUF_F16 = f'{WORK}/dizercore-f16.gguf'
GGUF_Q4 = f'{WORK}/dizercore-q4_k_m.gguf'
LLAMA_DIR = f'{WORK}/llama.cpp'

MAXLEN = 1024
EPOCHS = 1
EXAMPLES = 15000
LR = 2e-4

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print('=== Tokenizer + dataset ===', flush=True)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from datasets import load_dataset
raw = load_dataset('json', data_files=DATASET, split='train')

def to_text(ex):
    if ex.get('input'):
        return {'text': f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"}
    return {'text': f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"}

raw = raw.map(to_text, remove_columns=raw.column_names)
raw = raw.shuffle(seed=42).select(range(min(EXAMPLES, len(raw))))
tok_ds = raw.map(lambda b: tok(b['text'], truncation=True, max_length=MAXLEN),
                 batched=True, remove_columns=['text'])
print(f'Training on {len(tok_ds)} examples', flush=True)

print('=== Training ===', flush=True)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
))
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ADAPTER,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=True,
        logging_steps=10,
        optim='paged_adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        save_steps=250,
        save_total_limit=2,
        report_to='none',
        gradient_checkpointing=True,
    ),
    train_dataset=tok_ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)
trainer.train()
model.save_pretrained(ADAPTER)
print(f'Adapter saved: {ADAPTER}', flush=True)

print('=== Merging on fresh fp16 base ===', flush=True)
del model, trainer
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
del merged, base
gc.collect(); torch.cuda.empty_cache()
print('fp16 model saved', flush=True)

print('=== Cloning llama.cpp ===', flush=True)
if not os.path.isdir(LLAMA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp.git', LLAMA_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gguf', 'sentencepiece', 'protobuf'], check=True)

print('=== Converting to fp16 GGUF ===', flush=True)
subprocess.run([sys.executable, f'{LLAMA_DIR}/convert_hf_to_gguf.py',
                MERGED, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)
print(f'fp16 GGUF: {os.path.getsize(GGUF_F16)/1024/1024:.0f} MB', flush=True)

print('=== Building llama-quantize ===', flush=True)
subprocess.run(['cmake', '-B', f'{LLAMA_DIR}/build', '-S', LLAMA_DIR,
                '-DLLAMA_CURL=OFF', '-DGGML_NATIVE=OFF'], check=True)
subprocess.run(['cmake', '--build', f'{LLAMA_DIR}/build',
                '--target', 'llama-quantize', '-j', '4'], check=True)

print('=== Quantizing to Q4_K_M ===', flush=True)
subprocess.run([f'{LLAMA_DIR}/build/bin/llama-quantize',
                GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
print('=== DONE ===', flush=True)
print(f'GGUF: {size_mb:.0f} MB', flush=True)

import shutil
DRIVE_OUT = '/content/drive/MyDrive/dizercore/dizercore-q4_k_m.gguf'
shutil.copy(GGUF_Q4, DRIVE_OUT)
print(f'Copied to Drive: {DRIVE_OUT}', flush=True)

from IPython.display import FileLink, display
print('\nClick below to download:', flush=True)
display(FileLink(GGUF_Q4, result_html_prefix='Download: '))